# 第 6 周练习：按类别微调定价模型（Category-Specific Fine-Tuning）

## 练习目标

微调 **GPT-4.1-nano**，让模型按品类（电子、玩具、汽车等）估计产品价格，并比较**品类专家模型**与**通用模型**。

## 实验步骤

1. 加载数据集，按 `category` 分组  
2. 在混合品类上微调**通用模型**（基线）  
3. 对 2–3 个数据充足的品类微调**专家模型**  
4. 用可视化对比各品类上专家 vs 通用的 MAE  

## 和本课 Week 6 的关系

| 概念 | 本练习 |
|------|--------|
| JSONL 微调格式 | `messages` = user 询价 + assistant 价格 |
| 通用 vs 专家 | 混合采样 vs 单品类切片 |
| 路由推理 | 有专家用专家，否则回退通用 |
| 评估 | 按品类 MAE + `evaluate()` |


In [ ]:
# ========== 导入：路径、数据、OpenAI、定价工具、绘图 ==========

# os：读环境变量
import os
# Path：创建 jsonl 目录、拼文件路径
from pathlib import Path
# json：序列化 messages 进 JSONL
import json
# defaultdict：按类别聚合成列表
from collections import defaultdict
# load_dotenv：加载 .env 密钥
from dotenv import load_dotenv
# login：Hugging Face 登录（拉 items_lite）
from huggingface_hub import login
# OpenAI：微调与推理客户端
from openai import OpenAI
# Item：课程定价数据对象；from_hub 拉 train/val/test
from pricer.items import Item
# evaluate：课程统一评估入口（完整测试集可选）
from pricer.evaluator import evaluate
# matplotlib / pandas：条形图与结果表
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# ========== 环境、客户端、加载 items_lite、按类别分组 ==========

# 加载环境变量
load_dotenv(override=True)
# HF token：拉 Hub 数据集
hf_token = os.environ["HF_TOKEN"]
login(hf_token, add_to_git_credential=True)

# 微调需要 OpenAI API；作业进度见 https://platform.openai.com/finetune
# OpenRouter 不支持微调（只适合推理）
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# 可选：OpenRouter 仅用于基座模型零样本对比；ft:... 模型只能走 OpenAI
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
openrouter_client = (
    OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
    if OPENROUTER_API_KEY
    else None
)

# 使用 items_lite（含 LLM 生成的 summary）做微调输入
username = "ed-donner"
dataset = f"{username}/items_lite"
train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

# 把 Item 列表按 category 字段分桶
def by_category(items):
    d = defaultdict(list)
    for item in items:
        d[item.category].append(item)
    return dict(d)

# 三个划分各自一份「按类字典」
train_by_cat = by_category(train)
val_by_cat = by_category(val)
test_by_cat = by_category(test)

# 打印各类 train/val/test 数量，判断哪些类够做专家微调
for cat in sorted(train_by_cat.keys()):
    print(f"{cat}: train={len(train_by_cat[cat])}, val={len(val_by_cat[cat])}, test={len(test_by_cat[cat])}")


## 1. 可视化：每个类别的训练样本数

先看各类样本量分布，再决定哪些品类适合训「专家模型」。


In [ ]:
# ========== 条形图：各类别训练集规模 ==========

# 横轴类别名（排序），纵轴条数
categories = sorted(train_by_cat.keys())
counts = [len(train_by_cat[c]) for c in categories]

# 画布与柱状图
plt.figure(figsize=(12, 5))
plt.bar(categories, counts, color="steelblue")
plt.title("Training Items per Category")
plt.xlabel("Category")
plt.ylabel("Count")
# 类别名较长时旋转，避免重叠
plt.xticks(rotation=45, ha="right")
# 柱顶标注具体数字
for i, v in enumerate(counts):
    plt.text(i, v, str(v), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


## 2. 选择类别并准备微调数据

只保留至少有 **50** 条训练、**10** 条验证的类别。每个模型使用 **100** 条训练 / **25** 条验证（控制费用）。


In [ ]:
# ========== 阈值与筛选：数据够用的类别列表 ==========

# 最低样本门槛 + 每模型实际使用规模
MIN_TRAIN = 50
MIN_VAL = 10
TRAIN_SIZE = 100
VAL_SIZE = 25

# 同时满足 train/val 门槛的类别才进入后续实验
selected_categories = [
    cat for cat in sorted(train_by_cat.keys())
    if len(train_by_cat[cat]) >= MIN_TRAIN and len(val_by_cat[cat]) >= MIN_VAL
]
print(f"Selected categories: {selected_categories}")


In [ ]:
# ========== JSONL 辅助：与 day5 相同的 messages 格式 ==========

# 构造一条定价对话；category 非空时把品类写进询价句
def messages_for(item, category=None):
    # 默认通用询价 prompt（英文原样，禁止翻译）
    prompt = f"Estimate the price of this product. Respond with the price, no explanation"
    if category:
        # 专家模型：prompt 点名品类（下划线换成空格更可读）
        prompt = f"Estimate the price of this {category.replace('_', ' ')} product. Respond with the price, no explanation"
    return [
        {"role": "user", "content": f"{prompt}\n\n{item.summary}"},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

# 多条样本拼成 JSONL 文本（每行一个 JSON）
def make_jsonl(items, category=None):
    result = ""
    for item in items:
        msgs = messages_for(item, category)
        # 手工拼接 "messages" 键，保持与原代码一致
        result += '{"messages": ' + json.dumps(msgs) + "}\n"
    return result.strip()

# 确保父目录存在后写入文件
def write_jsonl(items, filename, category=None):
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    with open(filename, "w") as f:
        f.write(make_jsonl(items, category))


## 3. 创建 JSONL 文件

- **通用模型：** 从所有入选类别混合抽样 100 条  
- **类别专家：** 如 Electronics、Toys and Games 等，每类各 100 条  


In [ ]:
# ========== 通用模型数据：各类均衡抽样后截到 TRAIN_SIZE ==========

# 固定随机种子，保证可复现
import random
random.seed(42)

# 从每个 selected 类别抽 roughly 均分的样本，再打乱截断
general_train = []
per_cat = TRAIN_SIZE // len(selected_categories)
for cat in selected_categories:
    general_train.extend(random.sample(train_by_cat[cat], min(per_cat, len(train_by_cat[cat]))))
random.shuffle(general_train)
general_train = general_train[:TRAIN_SIZE]
# 验证集直接取全局 val 前 VAL_SIZE 条（与原逻辑一致）
general_val = val[:VAL_SIZE]

# 写出通用模型 JSONL（不带 category 特化 prompt）
write_jsonl(general_train, "jsonl/general_train.jsonl")
write_jsonl(general_val, "jsonl/general_val.jsonl")
print(f"General: {len(general_train)} train, {len(general_val)} val")


In [ ]:
# ========== 专家模型数据：只取前 2 个品类以控制费用 ==========

# 成本控制：先对前两个 selected 类别各训一个专家
SPECIALIST_CATEGORIES = selected_categories[:2]
print(f"Fine-tuning specialists for: {SPECIALIST_CATEGORIES}")

# 每类写 train/val JSONL；category=cat 会启用品类特化 prompt
for cat in SPECIALIST_CATEGORIES:
    ct = train_by_cat[cat][:TRAIN_SIZE]
    cv = val_by_cat[cat][:VAL_SIZE]
    write_jsonl(ct, f"jsonl/{cat}_train.jsonl", category=cat)
    write_jsonl(cv, f"jsonl/{cat}_val.jsonl", category=cat)
    print(f"  {cat}: {len(ct)} train, {len(cv)} val")


## 4. 上传文件并启动微调作业

先上传 JSONL，再 `fine_tuning.jobs.create`。通用模型与专家模型共用同一基础模型与超参。


In [ ]:
# ========== 通用模型：上传 JSONL + 创建微调作业 ==========

# 基础模型与超参（保持原 id / 数值）
BASE_MODEL = "gpt-4.1-nano-2025-04-14"
HYPERPARAMS = {"n_epochs": 1, "batch_size": 1}

# 上传通用 train / val 文件
with open("jsonl/general_train.jsonl", "rb") as f:
    general_train_file = client.files.create(file=f, purpose="fine-tune")
with open("jsonl/general_val.jsonl", "rb") as f:
    general_val_file = client.files.create(file=f, purpose="fine-tune")

# 启动通用微调；suffix 便于在模型名里识别
general_job = client.fine_tuning.jobs.create(
    training_file=general_train_file.id,
    validation_file=general_val_file.id,
    model=BASE_MODEL,
    seed=42,
    hyperparameters=HYPERPARAMS,
    suffix="pricer-general",
)
print(f"General job: {general_job.id}")


In [ ]:
# ========== 专家模型：逐品类上传并创建作业 ==========

# cat → FineTuningJob，后面轮询用
specialist_jobs = {}
for cat in SPECIALIST_CATEGORIES:
    # 上传该品类 train/val
    with open(f"jsonl/{cat}_train.jsonl", "rb") as f:
        tf = client.files.create(file=f, purpose="fine-tune")
    with open(f"jsonl/{cat}_val.jsonl", "rb") as f:
        vf = client.files.create(file=f, purpose="fine-tune")
    # suffix 截断品类名，避免过长
    job = client.fine_tuning.jobs.create(
        training_file=tf.id,
        validation_file=vf.id,
        model=BASE_MODEL,
        seed=42,
        hyperparameters=HYPERPARAMS,
        suffix=f"pricer-{cat.lower()[:20]}",
    )
    specialist_jobs[cat] = job
    print(f"{cat}: {job.id}")


## 5. 等待作业完成

轮询 `jobs.retrieve`，直到 `succeeded`（拿到 `fine_tuned_model`）或 `failed`。


In [ ]:
# ========== 轮询微调作业直到成功或失败 ==========

# time.sleep：避免打爆 API 的秒级忙等
import time

# 阻塞等待单个 job；成功返回微调模型 id
def poll_until_done(job_id):
    while True:
        j = client.fine_tuning.jobs.retrieve(job_id)
        status = j.status
        if status == "succeeded":
            return j.fine_tuned_model
        if status == "failed":
            raise RuntimeError(f"Job {job_id} failed")
        print(f"  {job_id}: {status}")
        time.sleep(30)

# 先等通用模型
general_model = poll_until_done(general_job.id)
print(f"General model: {general_model}")

# 再逐个等专家模型
specialist_models = {}
for cat, job in specialist_jobs.items():
    specialist_models[cat] = poll_until_done(job.id)
    print(f"{cat}: {specialist_models[cat]}")


## 6. 评估：专家 vs 通用

- **通用模型：** 对所有商品都用  
- **专家路由：** 商品类别有对应专家则用之，否则回退通用  


In [ ]:
# ========== 推理封装：通用定价器 + 专家路由器 ==========

# 测试时只用 user 消息（不带 assistant 真值）
def test_messages_for(item):
    return [{"role": "user", "content": f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"}]

# 调用指定模型，限制短输出（价格数字）
def predict(item, model_name):
    r = client.chat.completions.create(
        model=model_name,
        messages=test_messages_for(item),
        max_tokens=7,
    )
    return r.choices[0].message.content

# 始终走通用微调模型
def general_pricer(item):
    return predict(item, general_model)

# 有品类专家则用专家，否则回退通用
def specialist_router_pricer(item):
    # 有专家模型则用之，否则回退通用模型。
    model = specialist_models.get(item.category, general_model)
    return predict(item, model)


In [ ]:
# ========== MAE 工具：解析价格字符串 + 线程池批量预测 ==========

# re：从模型回复里抠数字；tqdm：进度条；ThreadPoolExecutor：并行打 API
import re
from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor

# 把 "$1,234.56" 这类文本收成 float；失败则 0
def post_process(value):
    if isinstance(value, str):
        value = value.replace("$", "").replace(",", "")
        m = re.search(r"[-+]?\d*\.\d+|\d+", value)
        return float(m.group()) if m else 0
    return float(value)

# 对一批 items 算平均绝对误差（Mean Absolute Error）
def mae_on_items(predictor, items, workers=5):
    errors = []
    with ThreadPoolExecutor(max_workers=workers) as ex:
        # ex.map 并行调用 predictor；与 items 对齐后算 |guess - price|
        for guess, item in zip(tqdm(ex.map(predictor, items), total=len(items), desc="Predicting"), items):
            g = post_process(guess)
            errors.append(abs(g - item.price))
    return sum(errors) / len(errors) if errors else 0


In [ ]:
# ========== 按品类对比 MAE：通用 vs 专家 ==========

# 每类只评估前 EVAL_SIZE 条，加快迭代
EVAL_SIZE = 50
results = []

for cat in SPECIALIST_CATEGORIES:
    # 取该品类测试子集
    test_items = test_by_cat.get(cat, [])[:EVAL_SIZE]
    if not test_items:
        continue
    # 通用模型 MAE
    mae_gen = mae_on_items(general_pricer, test_items)
    # 对应专家模型（可能尚未就绪）
    specialist_model = specialist_models.get(cat)
    mae_spec = mae_on_items(lambda i: predict(i, specialist_model), test_items) if specialist_model else None
    # improvement > 0 表示专家更好（MAE 更低）
    results.append({
        "category": cat,
        "n": len(test_items),
        "MAE (general)": mae_gen,
        "MAE (specialist)": mae_spec,
        "improvement": (mae_gen - mae_spec) if mae_spec else 0,
    })

# 用 DataFrame 方便展示与作图
results_df = pd.DataFrame(results)
results_df


## 7. 可视化：各类别上通用 MAE vs 专家 MAE

并排柱状图：柱越低越好；若专家柱更低，说明品类特化有效。


In [ ]:
# ========== 并排柱状图：General vs Specialist MAE ==========

df = results_df
x = range(len(df))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
# 左柱通用、右柱专家
ax.bar([i - w/2 for i in x], df["MAE (general)"], w, label="General", color="steelblue")
ax.bar([i + w/2 for i in x], df["MAE (specialist)"], w, label="Specialist", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(df["category"], rotation=45, ha="right")
ax.set_ylabel("Mean Absolute Error ($)")
ax.set_title("Category-Specific Fine-Tuning: General vs Specialist MAE")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 8. 完整测试集评估（可选）

用专家路由器在更大测试子集上调用课程自带的 `evaluate()`（会产生较多 API 调用）。


In [ ]:
# ========== 可选：在 200 条测试样本上评估专家路由器 ==========

# evaluate 内部会算误差分布等；size=200 控制调用量
evaluate(specialist_router_pricer, test, size=200)


## 报告摘要

- **通用模型：** 在混合品类上训练，作为比较基线。  
- **专家模型：** 各自只在单一品类上训练，预期在该品类 MAE 更低。  
- **如何读结果：** 看条形图；`improvement > 0` 表示该品类上专家优于通用。  
